<a href="https://colab.research.google.com/github/shivashree2121-gif/AI-Resume-Matching-System/blob/main/backend.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install streamlit pypdf python-docx nltk scikit-learn pandas numpy matplotlib seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 61.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.1/385.1 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 93.5 MB/s eta 0:00:00


In [2]:
import re
import string
import numpy as np
import pandas as pd

from pypdf import PdfReader
from docx import Document

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import nltk
from nltk.corpus import stopwords

nltk.download("stopwords")

STOP_WORDS = set(stopwords.words("english"))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [3]:
def extract_docx_text(file_path):

    document = Document(file_path)

    text = ""

    for paragraph in document.paragraphs:
        text += paragraph.text + "\n"

    return text

In [4]:
def extract_resume_text(file_path):

    if file_path.lower().endswith(".pdf"):
        return extract_pdf_text(file_path)

    elif file_path.lower().endswith(".docx"):
        return extract_docx_text(file_path)

    else:
        raise ValueError("Only PDF and DOCX files are supported.")

In [5]:
def clean_text(text):

    text = text.lower()

    text = re.sub(r'http\S+|www\S+', ' ', text)

    text = re.sub(r'\S+@\S+', ' ', text)

    text = re.sub(r'\d+', ' ', text)

    text = text.translate(
        str.maketrans('', '', string.punctuation)
    )

    words = text.split()

    words = [
        word for word in words
        if word not in STOP_WORDS
    ]

    return " ".join(words)

In [7]:
skills = [

    # Programming
    "python",
    "java",
    "r",
    "c++",

    # Database
    "sql",
    "mysql",
    "postgresql",
    "mongodb",

    # Data Analysis
    "excel",
    "power bi",
    "tableau",
    "pandas",
    "numpy",

    # Machine Learning
    "machine learning",
    "deep learning",
    "scikit learn",
    "tensorflow",
    "pytorch",

    # Statistics
    "statistics",
    "hypothesis testing",
    "regression",
    "probability",

    # Other
    "data analysis",
    "data visualization",
    "nlp",
    "artificial intelligence"
]

In [8]:
def extract_skills(text):

    text = text.lower()

    found_skills = []

    for skill in skills:

        if skill in text:
            found_skills.append(skill)

    return sorted(set(found_skills))

In [10]:
def calculate_similarity(resume_text, job_text):

    documents = [
        resume_text,
        job_text
    ]

    vectorizer = TfidfVectorizer()

    tfidf_matrix = vectorizer.fit_transform(documents)

    similarity = cosine_similarity(
        tfidf_matrix[0:1],
        tfidf_matrix[1:2]
    )

    return similarity[0][0] * 100

In [11]:
def resume_match(resume_text, job_description):

    cleaned_resume = clean_text(resume_text)

    cleaned_job = clean_text(job_description)

    resume_skills = extract_skills(resume_text)

    job_skills = extract_skills(job_description)

    matched_skills = list(
        set(resume_skills) &
        set(job_skills)
    )

    missing_skills = list(
        set(job_skills) -
        set(resume_skills)
    )

    similarity_score = calculate_similarity(
        cleaned_resume,
        cleaned_job
    )

    if len(job_skills) > 0:

        skill_score = (
            len(matched_skills) /
            len(job_skills)
        ) * 100

    else:

        skill_score = 0

    # Weighted final score
    final_score = (
        similarity_score * 0.6 +
        skill_score * 0.4
    )

    return {
        "Similarity Score": round(similarity_score, 2),
        "Skill Score": round(skill_score, 2),
        "Final Match Score": round(final_score, 2),
        "Matched Skills": sorted(matched_skills),
        "Missing Skills": sorted(missing_skills)
    }

In [13]:
def get_recommendation(score):

    if score >= 80:
        return "Excellent Match"

    elif score >= 65:
        return "Good Match"

    elif score >= 50:
        return "Moderate Match"

    else:
        return "Low Match"

In [14]:
score = result["Final Match Score"]

print(get_recommendation(score))

Good Match


In [15]:
import streamlit as st
import re
import string

from pypdf import PdfReader
from docx import Document

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import nltk
from nltk.corpus import stopwords

nltk.download("stopwords")

STOP_WORDS = set(stopwords.words("english"))

skills = [
    "python",
    "java",
    "r",
    "c++",
    "sql",
    "mysql",
    "postgresql",
    "mongodb",
    "excel",
    "power bi",
    "tableau",
    "pandas",
    "numpy",
    "machine learning",
    "deep learning",
    "scikit learn",
    "tensorflow",
    "pytorch",
    "statistics",
    "hypothesis testing",
    "regression",
    "probability",
    "data analysis",
    "data visualization",
    "nlp",
    "artificial intelligence"
]


def extract_pdf_text(uploaded_file):

    reader = PdfReader(uploaded_file)

    text = ""

    for page in reader.pages:

        page_text = page.extract_text()

        if page_text:
            text += page_text + "\n"

    return text


def extract_docx_text(uploaded_file):

    document = Document(uploaded_file)

    text = ""

    for paragraph in document.paragraphs:

        text += paragraph.text + "\n"

    return text


def extract_text(uploaded_file):

    if uploaded_file.name.lower().endswith(".pdf"):

        return extract_pdf_text(uploaded_file)

    elif uploaded_file.name.lower().endswith(".docx"):

        return extract_docx_text(uploaded_file)

    else:

        return ""


def clean_text(text):

    text = text.lower()

    text = re.sub(
        r'http\S+|www\S+',
        ' ',
        text
    )

    text = re.sub(
        r'\S+@\S+',
        ' ',
        text
    )

    text = re.sub(
        r'\d+',
        ' ',
        text
    )

    text = text.translate(
        str.maketrans(
            '',
            '',
            string.punctuation
        )
    )

    words = text.split()

    words = [
        word for word in words
        if word not in STOP_WORDS
    ]

    return " ".join(words)


def extract_skills(text):

    text = text.lower()

    found = []

    for skill in skills:

        if skill in text:

            found.append(skill)

    return sorted(set(found))


def calculate_similarity(resume, job):

    vectorizer = TfidfVectorizer()

    matrix = vectorizer.fit_transform(
        [resume, job]
    )

    similarity = cosine_similarity(
        matrix[0:1],
        matrix[1:2]
    )

    return similarity[0][0] * 100


def match_resume(resume, job):

    resume_clean = clean_text(resume)

    job_clean = clean_text(job)

    resume_skills = extract_skills(resume)

    job_skills = extract_skills(job)

    matched = list(
        set(resume_skills) &
        set(job_skills)
    )

    missing = list(
        set(job_skills) -
        set(resume_skills)
    )

    similarity = calculate_similarity(
        resume_clean,
        job_clean
    )

    if len(job_skills) > 0:

        skill_score = (
            len(matched) /
            len(job_skills)
        ) * 100

    else:

        skill_score = 0

    final_score = (
        similarity * 0.6 +
        skill_score * 0.4
    )

    return (
        similarity,
        skill_score,
        final_score,
        matched,
        missing
    )


# -----------------------------
# STREAMLIT UI
# -----------------------------

st.set_page_config(
    page_title="Resume Matcher",
    page_icon="📄",
    layout="wide"
)

st.title("📄 AI Resume–Job Matching System")

st.write(
    "Upload your resume and enter a job description "
    "to calculate the matching score."
)

resume_file = st.file_uploader(
    "Upload Resume",
    type=["pdf", "docx"]
)

job_description = st.text_area(
    "Paste Job Description",
    height=250
)


if st.button("🚀 Analyze Resume"):

    if resume_file is None:

        st.error(
            "Please upload your resume."
        )

    elif not job_description.strip():

        st.error(
            "Please enter a job description."
        )

    else:

        resume_text = extract_text(
            resume_file
        )

        similarity, skill_score, final_score, matched, missing = match_resume(
            resume_text,
            job_description
        )

        st.success(
            "Resume analyzed successfully!"
        )

        col1, col2, col3 = st.columns(3)

        col1.metric(
            "Text Similarity",
            f"{similarity:.2f}%"
        )

        col2.metric(
            "Skill Match",
            f"{skill_score:.2f}%"
        )

        col3.metric(
            "Overall Match",
            f"{final_score:.2f}%"
        )

        st.subheader("✅ Matched Skills")

        if matched:

            st.write(
                ", ".join(matched)
            )

        else:

            st.write(
                "No matching skills found."
            )

        st.subheader("❌ Missing Skills")

        if missing:

            st.write(
                ", ".join(missing)
            )

        else:

            st.write(
                "No major missing skills!"
            )

        st.subheader("📊 Recommendation")

        if final_score >= 80:

            st.success(
                "Excellent Match — Strongly recommended."
            )

        elif final_score >= 65:

            st.info(
                "Good Match — Consider applying."
            )

        elif final_score >= 50:

            st.warning(
                "Moderate Match — Improve missing skills."
            )

        else:

            st.error(
                "Low Match — Significant skill gaps detected."
            )

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
2026-08-28 10:15:59.221 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-28 10:15:59.225 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-28 10:15:59.615 
  command:

    streamlit run /usr/local/lib/python3.13/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-08-28 10:15:59.616 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-28 10:15:59.619 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-28 10:15:59.621 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in 